# [Open in Colab](https://colab.research.google.com/github/dmochow/BME-i9400-2026/blob/main/notebooks/meeting06_linear_regression.ipynb)

# BME i9400 — Meeting 6
## Linear Regression

**Fall 2026 · Monday, September 28**

Today's lecture is about regression, which is the prediction of a continuous quantity from a set of
measurements. The lecture develops the least squares solution of the linear model, first as a
minimisation problem and then as maximum likelihood under a model of the noise.
**Agenda**

1. The regression problem
2. The linear model
3. The least squares criterion
4. The normal equations
5. The noise model
6. Assessing a fit
7. Connection to the course

---
## 1 — The regression problem

A dataset for regression consists of a feature matrix and a target vector:

$$X \in \mathbb{R}^{n \times d}, \qquad y \in \mathbb{R}^{n}$$

- $X$ is the matrix of Meeting 4: $n$ samples as rows, $d$ features as columns
- $y$ is the **target**: one number per sample, which the model is asked to predict

> A **target** (also called the *response*, the *outcome*, or the *label*) is the quantity to be
> predicted. In regression the target is a continuous number — a concentration, a pressure, a
> score. In classification, from Meeting 8 onward, it is a category.
The learning problem is to find a function $f$ such that $f(x_i) \approx y_i$ for every sample in
the dataset and, more importantly, such that $f(x) \approx y$ for new samples that were not used to
find $f$. The second requirement is called **generalisation**, and section 6 returns to it.

### The running example

The dataset used today and in Wednesday's studio is from Efron, Hastie, Johnstone and Tibshirani,
"Least Angle Regression", *Annals of Statistics*, 2004. It is distributed with scikit-learn, so no
download is needed.
- $n = 442$ patients with diabetes
- $d = 10$ baseline measurements:
    - age (years)
    - sex (coded 1 or 2)
    - body mass index, BMI (kg/m²)
    - average blood pressure, BP (mm Hg)
    - six blood serum measurements: total cholesterol (`s1`), LDL cholesterol (`s2`), HDL
      cholesterol (`s3`), the ratio of total cholesterol to HDL (`s4`), the logarithm of serum
      triglycerides (`s5`), and blood glucose (`s6`)
- $y$ is a quantitative measure of disease progression one year after the baseline visit

Regressing progression on the baseline measurements asks which of the measurements, taken today,
predict how far the disease will have progressed in a year, and by how much.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
plt.rcParams.update({"font.size": 12, "figure.dpi": 110})

data     = load_diabetes(scaled=False, as_frame=True)     # scaled=False keeps the original units
features = ["age", "sex", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6"]

X = data.frame[features].values        # n x d feature matrix
y = data.frame["target"].values        # n targets: disease progression at one year
print(f"X has shape {X.shape}  ->  n = {X.shape[0]} patients, d = {X.shape[1]} features")
print(f"y has shape {y.shape}")
print(f"progression: mean {y.mean():.1f}, standard deviation {y.std(ddof=1):.1f}, "
      f"range {y.min():.0f} to {y.max():.0f}")
data.frame.head(4)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6), sharey=True)
for ax, j in zip(axes.ravel(), range(10)):
    r = np.corrcoef(X[:, j], y)[0, 1]                      # correlation coefficient, Meeting 4
    ax.scatter(X[:, j], y, s=8, alpha=0.4, color="#6e6878")
    ax.set_title(f"{features[j]}   (r = {r:+.2f})", fontsize=11)
    ax.grid(alpha=0.25)
for ax in axes[:, 0]:
    ax.set_ylabel("progression")
fig.suptitle("Disease progression against each of the ten baseline measurements", y=1.02)
plt.tight_layout(); plt.show()

Each panel plots the target against one feature, with the correlation coefficient in the title. BMI
and `s5` have the strongest linear relationships with progression; `sex` and `s4` take only a few
distinct values. No single feature comes close to determining the target: the scatter within every
panel is large.

Sections 2 to 5 develop the method using BMI alone, so that everything can be drawn. Section 6 uses
all ten features.

---
## 2 — The linear model

A linear model predicts the target as a weighted sum of the features plus a constant:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \cdots + w_d x_d$$

> $\hat{y}$, read "y hat", is the model's **prediction**. The hat distinguishes it from the observed
> value $y$.
>
> $w_1, \dots, w_d$ are the **weights**, also called **coefficients**. $w_j$ is the change in the
> prediction per unit increase in feature $j$, with the other features held fixed.
>
> $w_0$ is the **intercept** (or *bias*): the prediction when every feature is zero.

With BMI as the single feature the model is a line, $\hat{y} = w_0 + w_1 \cdot \text{BMI}$, and
$w_1$ is the predicted change in progression per unit of BMI.
### Absorbing the intercept

The intercept is handled by treating it as the weight on a feature that always equals 1. A column of
ones is placed in front of the features:

$$X = \begin{pmatrix} 1 & x_{11} & \cdots & x_{1d} \\ 1 & x_{21} & \cdots & x_{2d} \\ \vdots & \vdots & & \vdots \\ 1 & x_{n1} & \cdots & x_{nd} \end{pmatrix} \in \mathbb{R}^{n \times (d+1)},
\qquad
w = \begin{pmatrix} w_0 \\ w_1 \\ \vdots \\ w_d \end{pmatrix} \in \mathbb{R}^{d+1}$$

> With the column of ones included, $X$ is called the **design matrix**. From here on, $X$ means
> the design matrix, and $w$ includes the intercept as its first entry.
### Predictions as products

The prediction for sample $i$ is a dot product, and the predictions for all samples at once are a
matrix–vector product:

$$\hat{y}_i = x_i \cdot w = x_i^{\top} w, \qquad\qquad \hat{y} = X w$$

> $x_i^{\top} w$ is the dot product $x_i \cdot w$ written as a row vector times a column vector. Both
> notations are in common use.
>
> $Xw$ multiplies every row of $X$ by $w$. It computes all $n$ dot products in one operation and
> returns a vector of $n$ predictions.

From Meeting 4, a dot product with a fixed vector is a projection: the linear model projects every
sample onto the weight vector $w$ and uses the resulting coordinate as the prediction.

In [ ]:
bmi = X[:, 2]                                       # the BMI column
X1  = np.column_stack([np.ones(len(bmi)), bmi])     # design matrix: a column of ones, then BMI

print("design matrix, first four rows:")
print(X1[:4])
print("shape:", X1.shape, " ->  n = 442 rows, d + 1 = 2 columns\n")

w_guess = np.array([0.0, 6.0])                      # w0 = 0, w1 = 6: a line through the origin
y_hat   = X1 @ w_guess                              # one prediction per patient
print("predictions from w_guess, first four patients:", np.round(y_hat[:4], 1))
print("observed progression,     first four patients:", y[:4])

---
## 3 — The least squares criterion

For a given weight vector $w$, sample $i$ has a **residual**: the difference between what was
observed and what the model predicted.

$$r_i = y_i - \hat{y}_i = y_i - x_i^{\top} w$$

> A **residual** is the part of the target that the model did not explain. A positive residual means
> the model predicted too low; a negative one means it predicted too high.
Least squares chooses $w$ to make the residuals small, measured by the sum of their squares:

$$L(w) = \sum_{i=1}^{n} r_i^2 = \sum_{i=1}^{n} \left(y_i - x_i^{\top} w\right)^2 = \|y - Xw\|^2$$

> $L(w)$ is the **loss function**, or *objective*. It assigns one number to every candidate $w$: the
> total squared error that $w$ makes on the data. Fitting a model means finding the $w$ with the
> smallest loss.
>
> $\|y - Xw\|^2$ is the squared norm from Meeting 4 — the sum of the squares of the entries of the
> residual vector $r = y - Xw$. The sum and the norm are two ways of writing the same quantity.
Two reasons for squaring the residuals, rather than taking absolute values or some other function of
them:

- squaring makes the loss a smooth function of $w$, so its minimum can be found by calculus
  (section 4)
- squaring is what a Gaussian model of the noise implies (section 5)

The least squares estimate is the minimiser:

$$w^{*} = \arg\min_{w} \|y - Xw\|^2$$

> $\arg\min_w$ means "the value of $w$ at which the minimum is attained", as opposed to $\min_w$,
> which is the minimum value itself.

$L$ is a quadratic function of $w$, so its graph is a bowl, and it has exactly one minimum whenever
the bowl curves upward in every direction. Section 4 states when that condition holds.

In [ ]:
def loss(w, X=X1, y=y):
    """Sum of squared residuals for the weight vector w."""
    r = y - X @ w
    return np.sum(r**2)

candidates = {"guess A": np.array([0.0, 6.0]),
              "guess B": np.array([-200.0, 14.0]),
              "guess C": np.array([-118.0, 10.2])}

for name, w in candidates.items():
    print(f"{name}: w0 = {w[0]:7.1f}, w1 = {w[1]:5.1f},  L = {loss(w)/1e6:.2f} million")
rng   = np.random.default_rng(0)
shown = rng.choice(len(y), 60, replace=False)          # a subset of patients, so the segments are visible

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
for ax, (name, w) in zip(axes, candidates.items()):
    grid = np.array([17, 43])
    ax.plot(grid, w[0] + w[1] * grid, color="#c1272d", lw=2)
    ax.scatter(bmi[shown], y[shown], s=18, color="#6e6878", zorder=3)
    ax.vlines(bmi[shown], X1[shown] @ w, y[shown], color="#2b6cb0", lw=1, alpha=0.8)   # the residuals
    ax.set_title(f"{name}:  $L$ = {loss(w)/1e6:.2f} million")
    ax.set_xlabel("BMI"); ax.grid(alpha=0.25)
axes[0].set_ylabel("progression")
plt.tight_layout(); plt.show()

In [ ]:
# The loss as a function of the weights, evaluated on a grid of (w0, w1) values.
bmi_c = bmi - bmi.mean()                               # centered BMI, for the right-hand panel
X1c   = np.column_stack([np.ones(len(bmi_c)), bmi_c])

def loss_grid(X, w0_grid, w1_grid):
    """Loss at every (w0, w1) pair: one row per w1 value, one column per w0 value."""
    out = np.empty((len(w1_grid), len(w0_grid)))
    for k, w1 in enumerate(w1_grid):
        r = y[None, :] - w0_grid[:, None] - w1 * X[None, :, 1]     # residuals for every w0 at once
        out[k] = np.sum(r**2, axis=1)
    return out

w1_grid = np.linspace(-6, 26, 321)
w0_raw  = np.linspace(-700, 500, 481)
w0_cen  = np.linspace(100, 205, 421)
L_raw   = loss_grid(X1,  w0_raw, w1_grid)
L_cen   = loss_grid(X1c, w0_cen, w1_grid)

k, j = np.unravel_index(np.argmin(L_raw), L_raw.shape)          # location of the smallest grid value
print(f"smallest loss on the grid: {L_raw[k, j]/1e6:.4f} million, at w0 = {w0_raw[j]:.1f}, w1 = {w1_grid[k]:.2f}")
levels = L_raw.min()/1e6 * np.array([1.001, 1.01, 1.03, 1.1, 1.3, 1.7, 2.5, 4, 7, 12, 20, 35])
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
axes[0].plot(w1_grid, L_raw[:, j]/1e6, color="#5b3a8e", lw=2.5)
axes[0].plot(w1_grid[k], L_raw[k, j]/1e6, "o", color="#c1272d", ms=9)
axes[0].set_xlabel(f"$w_1$   (with $w_0$ fixed at {w0_raw[j]:.0f})"); axes[0].set_ylabel("$L$  (millions)")
axes[0].set_title("Loss along one weight")

axes[1].contour(w0_raw, w1_grid, L_raw/1e6, levels=levels, cmap="viridis")
axes[1].plot(w0_raw[j], w1_grid[k], "o", color="#c1272d", ms=9)
axes[1].set_title("Loss over both weights, BMI as recorded")

kc, jc = np.unravel_index(np.argmin(L_cen), L_cen.shape)
axes[2].contour(w0_cen, w1_grid, L_cen/1e6, levels=levels, cmap="viridis")
axes[2].plot(w0_cen[jc], w1_grid[kc], "o", color="#c1272d", ms=9)
axes[2].set_title("Loss over both weights, BMI centered")
for ax in axes[1:]:
    ax.set_xlabel("$w_0$"); ax.set_ylabel("$w_1$")
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

The left panel fixes $w_0$ and varies $w_1$: a parabola with one minimum. The middle panel varies both
weights; the contours are ellipses, so the surface is a bowl with a single lowest point, marked in
red. The bowl is extremely elongated along one diagonal. Many combinations of $(w_0, w_1)$ give
nearly the same loss, because the intercept and the slope can trade off against each other when BMI
takes values near 26 rather than near 0: raising the slope and lowering the intercept moves the line
very little over the range of the data.
Centering BMI (right panel) removes the trade-off and makes the bowl round. The slope is unchanged,
and the intercept becomes the mean of $y$. The elongation matters for numerical accuracy and for
minimisation by gradient descent (Meeting 13); it does not change where the minimum is.

---
## 4 — The normal equations

The minimum of a smooth function is where its gradient is zero. Expanding the loss:

$$L(w) = \|y - Xw\|^2 = (y - Xw)^{\top}(y - Xw)$$

$$= y^{\top}y - y^{\top}Xw - w^{\top}X^{\top}y + w^{\top}X^{\top}Xw$$

$$= y^{\top}y - 2\,w^{\top}X^{\top}y + w^{\top}X^{\top}Xw$$

> $(Xw)^{\top} = w^{\top}X^{\top}$: transposing a product reverses the order of the factors.
>
> $y^{\top}Xw$ and $w^{\top}X^{\top}y$ are the same number — each is the transpose of the other, and
> a $1 \times 1$ matrix equals its transpose — which is why the two middle terms combine.
### The gradient

The gradient with respect to $w$ follows from two rules of vector calculus, applied term by term:

- $\nabla_w \left(w^{\top} b\right) = b$ for a constant vector $b$
- $\nabla_w \left(w^{\top} A w\right) = 2Aw$ for a constant symmetric matrix $A$

> The **gradient** $\nabla_w L$ is the vector of partial derivatives
> $\left(\partial L/\partial w_0,\; \partial L/\partial w_1,\; \dots,\; \partial L/\partial w_d\right)$.
> It points in the direction in which $L$ increases fastest, and it is the zero vector at a minimum.

Here $b = X^{\top}y$ and $A = X^{\top}X$, which is symmetric because
$(X^{\top}X)^{\top} = X^{\top}X$. Therefore

$$\nabla_w L = -2X^{\top}y + 2X^{\top}Xw$$
### Setting the gradient to zero

Dividing by 2 and rearranging:

$$X^{\top}X\,w = X^{\top}y$$

These are the **normal equations**: a system of $d+1$ linear equations in the $d+1$ unknowns
$w_0, \dots, w_d$. When the matrix $X^{\top}X$ is invertible, the system has exactly one solution,

$$\boxed{\;w^{*} = \left(X^{\top}X\right)^{-1} X^{\top} y\;}$$

> $X^{\top}X$ is a $(d+1) \times (d+1)$ matrix — the same product that formed the covariance matrix
> in Meeting 4, without the centering and the division by $n-1$. Its entry $(j, k)$ is the dot
> product of feature columns $j$ and $k$.
>
> $\left(X^{\top}X\right)^{-1}$ is its **matrix inverse**: the matrix that multiplies $X^{\top}X$ to
> give the identity. Multiplying both sides of the normal equations by it isolates $w$.
### The residuals at the solution

Moving both terms of the normal equations to one side gives an equivalent statement about the
residual vector $r = y - Xw^{*}$:

$$X^{\top}\left(y - Xw^{*}\right) = 0 \qquad\Longleftrightarrow\qquad X^{\top} r = 0$$

> $X^{\top} r$ is a vector of $d+1$ dot products, one between the residuals and each column of $X$.
> At the least squares solution every one of them is zero: the residuals have no remaining
> correlation with any feature. If they did, adjusting that feature's weight would reduce the loss
> further.

The first column of $X$ is the column of ones, so the first of these equations reads
$\sum_i r_i = 0$: **the residuals of a least squares fit with an intercept always sum to zero.**
### The one-feature case

With a single feature and an intercept, the normal equations can be solved by hand. Writing
$\bar{x}$ and $\bar{y}$ for the means of the feature and the target,

$$w_1^{*} = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2} = \frac{\operatorname{Cov}(x, y)}{\operatorname{Var}(x)},
\qquad\qquad w_0^{*} = \bar{y} - w_1^{*}\,\bar{x}$$

The slope is the covariance of feature and target divided by the variance of the feature, both from
section 4 of Meeting 4. The intercept places the line through the point $(\bar{x}, \bar{y})$.
### When $X^{\top}X$ is not invertible

$X^{\top}X$ is invertible exactly when the columns of $X$ are **linearly independent**. Two
situations break this:

- **fewer samples than unknowns**, $n < d + 1$: a $(d+1) \times (d+1)$ matrix built from only $n$
  rows cannot have full rank
- **exactly collinear features**: one column is a weighted sum of others, for example a measurement
  recorded twice in different units

> Columns are **linearly independent** when the only weighted sum of them that gives the zero vector
> is the one with all weights zero. If some other combination gives zero, then adding that combination
> to $w$ leaves $Xw$ unchanged — and the loss with it — so the minimum is not unique.
In either case the loss still has a minimum value, but infinitely many $w$ attain it. In practice the
failure is rarely exact: columns that are *nearly* dependent make $X^{\top}X$ *nearly* singular, and
the solution, although unique, becomes extremely sensitive to the particular sample. Section 5
quantifies this sensitivity, and Wednesday's studio introduces the standard remedy, which is to add a
small multiple of the identity matrix to $X^{\top}X$.

### Computing the solution

The formula says "invert $X^{\top}X$"; numerical practice does otherwise. Solving the linear system
$X^{\top}X\,w = X^{\top}y$ directly (`np.linalg.solve`) is faster and more accurate than forming
the inverse, and solving the least squares problem from $X$ itself (`np.linalg.lstsq`, which uses the
SVD of Meeting 4) is more accurate still, because forming $X^{\top}X$ squares the ratio of the largest
to the smallest singular value and so amplifies rounding error. Scikit-learn's `LinearRegression`
takes the third route.

In [ ]:
# Solve the normal equations for the BMI model, several ways.
XtX = X1.T @ X1                        # (d+1) x (d+1)
Xty = X1.T @ y                         # (d+1)
print("X^T X =\n", np.round(XtX, 1))
print("X^T y =", np.round(Xty, 1), "\n")

w_inv   = np.linalg.inv(XtX) @ Xty                      # the formula, literally
w_solve = np.linalg.solve(XtX, Xty)                     # solve the linear system instead
w_lstsq = np.linalg.lstsq(X1, y, rcond=None)[0]         # least squares from X directly, via the SVD

print(f"inverse formula    : w0 = {w_inv[0]:9.3f}, w1 = {w_inv[1]:.4f}")
print(f"np.linalg.solve    : w0 = {w_solve[0]:9.3f}, w1 = {w_solve[1]:.4f}")
print(f"np.linalg.lstsq    : w0 = {w_lstsq[0]:9.3f}, w1 = {w_lstsq[1]:.4f}")
# The one-feature formulas, and two checks on the solution.
w1_hand = np.cov(bmi, y, ddof=1)[0, 1] / np.var(bmi, ddof=1)
w0_hand = y.mean() - w1_hand * bmi.mean()
print(f"Cov/Var formulas   : w0 = {w0_hand:9.3f}, w1 = {w1_hand:.4f}")
print(f"grid search, section 3: w0 = {w0_raw[j]:.1f}, w1 = {w1_grid[k]:.2f}   <- limited by the grid spacing")
print(f"loss at the solution: {loss(w_solve)/1e6:.4f} million   (smallest grid value: {L_raw[k, j]/1e6:.4f} million)")

w_star   = w_solve
gradient = -2 * X1.T @ y + 2 * X1.T @ X1 @ w_star
residual = y - X1 @ w_star
print("\ngradient at the solution:", gradient, "  <- zero, up to rounding")
print("X^T r at the solution   :", X1.T @ residual, "  <- zero: the residuals sum to zero and are uncorrelated with BMI")

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression().fit(bmi.reshape(-1, 1), y)    # scikit-learn expects a 2-D feature array
print(f"scikit-learn : intercept = {model.intercept_:.3f}, coefficient = {model.coef_[0]:.4f}")
print(f"our solution : intercept = {w_star[0]:.3f}, coefficient = {w_star[1]:.4f}")
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(bmi, y, s=12, alpha=0.5, color="#6e6878")
grid = np.array([17, 43])
ax.plot(grid, w_star[0] + w_star[1] * grid, color="#c1272d", lw=2.5,
        label=f"$\\hat{{y}}$ = {w_star[0]:.1f} + {w_star[1]:.2f} BMI")
ax.plot(bmi.mean(), y.mean(), "o", color="#1a1a1a", ms=9, label="($\\bar{x}$, $\\bar{y}$)")
ax.set_xlabel("BMI (kg/m²)"); ax.set_ylabel("disease progression")
ax.set_title("The least squares line")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The fitted line predicts 10.2 additional units of progression per unit of BMI. It passes through the
point of means, as the one-feature formulas require, and the exact solution agrees with the grid
search of section 3 to the precision of the grid, without any search.

---
## 5 — The noise model

Sections 3 and 4 chose the squared loss and derived its consequences. This section gives a reason to
choose it: least squares is what results from assuming a specific model of how the data were
generated.

Assume that each target is a linear function of the features plus a random error:

$$y_i = x_i^{\top} w + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma^2), \quad \text{independently for each } i$$

> $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$ reads "$\varepsilon_i$ is drawn from a Gaussian
> (normal) distribution with mean 0 and variance $\sigma^2$". The **Gaussian distribution** is the
> bell curve; its density at a value $\varepsilon$ is
> $$p(\varepsilon) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{\varepsilon^2}{2\sigma^2}\right)$$
>
> **Independently** means that one patient's error says nothing about another's. The errors are then
> **i.i.d.**: independent and identically distributed.
>
> $\varepsilon_i$ collects everything that affects the target but is not in the model: unmeasured
> variables, measurement error, and biological variability. $\sigma$ is its typical size, in the
> units of $y$.
### The likelihood

Under this model $y_i$ is itself Gaussian, with mean $x_i^{\top} w$ and variance $\sigma^2$. Because
the errors are independent, the probability density of observing the whole dataset, for a given $w$,
is the product of the individual densities:

$$p(y \mid X, w) = \prod_{i=1}^{n} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(y_i - x_i^{\top} w)^2}{2\sigma^2}\right)$$

> Viewed as a function of $w$ with the data held fixed, this quantity is the **likelihood** of $w$:
> how probable the observed data would be if $w$ were the true weight vector. **Maximum likelihood**
> estimation chooses the $w$ that makes the observed data most probable.
### The log-likelihood

Products of many small numbers are awkward, so take the logarithm, which is an increasing function
and therefore does not move the maximum:

$$\log p(y \mid X, w) = \sum_{i=1}^{n} \left[ -\frac{1}{2}\log(2\pi\sigma^2) - \frac{(y_i - x_i^{\top} w)^2}{2\sigma^2} \right]$$

$$= -\frac{n}{2}\log(2\pi\sigma^2) \;-\; \frac{1}{2\sigma^2} \sum_{i=1}^{n} (y_i - x_i^{\top} w)^2$$

The first term does not depend on $w$. The second is $-\frac{1}{2\sigma^2}\,L(w)$: the least squares
loss, multiplied by a negative constant. Maximising the log-likelihood over $w$ is therefore the same
as minimising $L(w)$:

$$\boxed{\;\text{least squares} \;=\; \text{maximum likelihood under i.i.d. Gaussian noise}\;}$$
The logarithm of a Gaussian density is a parabola in the error, and a sum of parabolas is the squared
loss. A different noise distribution gives a different loss: errors with a Laplace distribution, for
example, lead to the sum of absolute residuals. The squared loss is the right choice exactly when the
Gaussian assumption is reasonable, and it is a poor choice when the errors have heavy tails, because a
single extreme residual then dominates the sum.

### The fitted weights are random

The observed $y$ contains one draw of the noise. A second cohort of patients would contain a
different draw, and the least squares solution computed from it would differ. Substituting the model
$y = Xw + \varepsilon$ into the solution,

$$w^{*} = (X^{\top}X)^{-1}X^{\top}(Xw + \varepsilon) = (X^{\top}X)^{-1}(X^{\top}X)\,w + (X^{\top}X)^{-1}X^{\top}\varepsilon = w + (X^{\top}X)^{-1}X^{\top}\varepsilon$$

so $w^{*}$ equals the true $w$ plus a term driven by the noise.
The noise term has mean zero, so least squares is **unbiased**, and its covariance is

$$\operatorname{Cov}(w^{*}) = \sigma^2 \left(X^{\top}X\right)^{-1}$$

> **Unbiased** means that the estimate is correct on average over repeated datasets: if the study
> were repeated many times, the fitted weights would scatter around the true weights rather than
> around some other value. Any single estimate is still off by an amount whose typical size is set by
> the covariance — the square root of its $j$-th diagonal entry is the standard deviation of $w_j^{*}$.

The covariance formula says where uncertainty in the weights comes from. It grows with the noise
$\sigma^2$; it shrinks with more data, because the entries of $X^{\top}X$ grow with $n$ and those of
the inverse shrink; and it **becomes very large when $X^{\top}X$ is nearly singular**, the collinear
case of section 4. The simulation below checks the formula, and Wednesday's studio makes the
collinear case concrete.

In [ ]:
# Take the fitted BMI line as the truth, add Gaussian noise of the estimated size, refit, and repeat.
n_obs, n_par = X1.shape
sigma_hat = np.sqrt(np.sum((y - X1 @ w_star)**2) / (n_obs - n_par))   # residual standard deviation;
print(f"estimated noise level: sigma_hat = {sigma_hat:.1f} progression units\n")   # n - 2 because two weights were fitted

rng = np.random.default_rng(6)

def simulate_slopes(X, w_true, sigma, reps=3000):
    """Refit the model on `reps` simulated datasets and return the fitted slopes."""
    slopes = np.empty(reps)
    for k in range(reps):
        y_sim = X @ w_true + rng.normal(0, sigma, size=len(X))     # the noise model, sampled once
        slopes[k] = np.linalg.solve(X.T @ X, X.T @ y_sim)[1]
    return slopes
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)
for ax, n in zip(axes, [442, 25]):
    rows   = rng.choice(n_obs, size=n, replace=False)              # a cohort of n patients
    Xn     = X1[rows]
    slopes = simulate_slopes(Xn, w_star, sigma_hat)
    theory = sigma_hat * np.sqrt(np.linalg.inv(Xn.T @ Xn)[1, 1])   # square root of the (1, 1) entry of Cov(w*)
    ax.hist(slopes, bins=50, density=True, color="#5b3a8e", alpha=0.8)
    ax.axvline(w_star[1], color="#c1272d", lw=2)
    ax.set_title(f"n = {n}:  standard deviation of the slope {slopes.std():.2f}   (formula {theory:.2f})")
    ax.set_xlabel("fitted slope $w_1$"); ax.grid(alpha=0.25)
    print(f"n = {n:3d}:  mean of the fitted slopes = {slopes.mean():.2f}  (true value {w_star[1]:.2f}),  "
          f"standard deviation = {slopes.std():.2f}  (formula {theory:.2f})")
axes[0].set_ylabel("density")
plt.tight_layout(); plt.show()

The fitted slopes are centered on the true value in both panels, which is unbiasedness, and their
spread matches $\sigma\sqrt{[(X^{\top}X)^{-1}]_{11}}$. With 25 patients the slope is uncertain by
about 3.4 units per unit of BMI, a third of its value, and a single small cohort can easily produce
a slope of 7 or of 14. Wednesday's studio addresses this variability directly.

---
## 6 — Assessing a fit

Fitting produces $w^{*}$. Whether the fit is any good is a separate question, answered by the
residuals and by the model's performance on data it has not seen.

### Error measures

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n} r_i^2, \qquad\qquad \text{RMSE} = \sqrt{\text{MSE}}$$

> **MSE** is the **mean squared error**: the loss divided by the number of samples. **RMSE**, its
> square root, is in the units of $y$. An RMSE of 54 means that the predictions are typically off by
> about 54 progression units. Under the noise model, the RMSE on the training data estimates
> $\sigma$, apart from the $n$ versus $n - (d+1)$ in the denominator.
$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2} = 1 - \frac{\|r\|^2}{\|y - \bar{y}\|^2}$$

> $R^2$, the **coefficient of determination**, compares the model's squared error with that of the
> simplest possible predictor, the constant $\bar{y}$. It is the fraction of the variance of $y$ that
> the model accounts for. $R^2 = 1$ is a perfect fit; $R^2 = 0$ is no better than predicting the
> mean; on new data $R^2$ can be negative.
### Training error and held-out error

The loss was minimised on the training data, so the training RMSE is an optimistic estimate of how
the model will perform on new patients: the weights were adjusted to these particular residuals,
noise included. The honest estimate comes from patients that played no part in the fit.

The standard procedure is to **split** the data before fitting: a training set, used to compute
$w^{*}$, and a held-out set, used only to evaluate it. Wednesday's studio does this throughout, and
Meetings 11 and 12 develop the procedure into cross-validation.

> **Held-out** data are samples set aside before fitting and never used to choose anything about the
> model — neither its weights nor, later in the course, its hyperparameters. Performance measured on
> them estimates performance on future patients.
### Residual plots

Under the noise model the residuals should look like i.i.d. Gaussian noise: no trend against the
fitted values, no dependence on any feature, and a roughly bell-shaped histogram. Structure in the
residuals is evidence that the model is missing something — a curved relationship, an interaction
between features, an omitted variable — or that the noise is not Gaussian.

In [ ]:
from sklearn.model_selection import train_test_split

# All ten features. Hold out 20% of the patients before fitting anything.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=9400)

def design(A):
    """Place the column of ones in front of the feature matrix A."""
    return np.column_stack([np.ones(len(A)), A])

Xd_train, Xd_test = design(X_train), design(X_test)
w_full = np.linalg.solve(Xd_train.T @ Xd_train, Xd_train.T @ y_train)
print(f"training patients: {len(y_train)}     held-out patients: {len(y_test)}\n")
print("fitted weights, in the units of each feature:")
for name, coef in zip(["intercept"] + features, w_full):
    print(f"  {name:<10} {coef:9.3f}")
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

def r_squared(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - y_true.mean())**2)

pred_train, pred_test = Xd_train @ w_full, Xd_test @ w_full

print(f"{'':<12}{'RMSE':>8}{'R^2':>8}")
print(f"{'training':<12}{rmse(y_train, pred_train):>8.1f}{r_squared(y_train, pred_train):>8.3f}")
print(f"{'held-out':<12}{rmse(y_test,  pred_test):>8.1f}{r_squared(y_test,  pred_test):>8.3f}")

In [ ]:
from scipy.stats import norm

resid_train = y_train - pred_train

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
axes[0].scatter(pred_test, y_test, s=18, color="#6e6878")
axes[0].plot([0, 350], [0, 350], "--", color="#c1272d", lw=1.5)
axes[0].set_xlabel("predicted progression"); axes[0].set_ylabel("observed progression")
axes[0].set_title(f"Held-out patients  (RMSE {rmse(y_test, pred_test):.1f})")

axes[1].scatter(pred_train, resid_train, s=12, alpha=0.6, color="#6e6878")
axes[1].axhline(0, color="#c1272d", lw=1.5)
axes[1].set_xlabel("fitted value"); axes[1].set_ylabel("residual")
axes[1].set_title("Residuals against fitted values (training)")

axes[2].hist(resid_train, bins=30, density=True, color="#5b3a8e", alpha=0.75)
grid = np.linspace(-200, 200, 300)
axes[2].plot(grid, norm(0, resid_train.std()).pdf(grid), color="#c1272d", lw=2,
             label=f"Gaussian, $\\sigma$ = {resid_train.std():.1f}")
axes[2].set_xlabel("residual"); axes[2].set_ylabel("density"); axes[2].legend()
axes[2].set_title("Residual distribution (training)")
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print(f"mean of the training residuals: {resid_train.mean():.2e}   <- zero, as section 4 requires")

Ten features account for about half of the variance of progression: $R^2 \approx 0.52$ on the
training patients and $0.48$ on the held-out patients, with an RMSE near 54 units on both. The two
RMSE values are close, so the model is not fitting noise; with 353 training patients and 11 weights
there is little room to. The residuals show no trend against the fitted values and are approximately
Gaussian, so the noise model is not contradicted.
The weights are in the units of their features, which makes their sizes hard to compare. `s5`, the
logarithm of triglycerides, has a weight near 85 because a one-unit change in a logarithm is a large
change; `age` has a weight near zero. The pair `s1` and `s2`, total cholesterol and LDL, is the case
to watch: the two are correlated at $r = 0.90$, and Wednesday's studio examines what that does to
their weights.

---
## 7 — Connection to the course

**The linear model is the baseline against which everything else is judged.** A learning outcome
of this course is to build a defensible baseline before adapting anything larger. For a continuous
target, the least squares fit of section 6 is that baseline, and a neural network that cannot beat it
has not learned anything the linear model did not.

**Loss, gradient, minimum.** The pattern of section 4 — write a loss, take its gradient, set it to
zero — is the pattern of every fitting procedure in this course. Least squares is the one case with a
closed-form answer. Logistic regression (Meeting 8) and neural networks (Meeting 17) produce
gradients that cannot be set to zero analytically and are minimised instead by following the gradient
downhill (Meeting 13).
**Maximum likelihood gives the loss.** Section 5 derived the squared loss from Gaussian noise.
Meeting 8 derives the loss for classification, the cross-entropy, from a Bernoulli model of the
labels by the same steps: write the likelihood, take the logarithm, negate.

**Variance of the estimate is the problem that regularisation addresses.** Section 5 showed that
the fitted weights scatter around the truth by an amount set by $\sigma^2 (X^{\top}X)^{-1}$, which is
large when the data are few or the features collinear. Wednesday's studio adds a penalty to the loss
that reduces this scatter at the cost of a small bias; Meeting 11 treats the trade-off in general.
## What to take away

- A linear model predicts $\hat{y} = Xw$, with the intercept absorbed as a column of ones in the
  design matrix
- Least squares minimises $L(w) = \|y - Xw\|^2$, a bowl-shaped function with a single minimum when
  the columns of $X$ are linearly independent
- The minimum satisfies the normal equations $X^{\top}Xw = X^{\top}y$, with solution
  $w^{*} = (X^{\top}X)^{-1}X^{\top}y$ — computed in practice by `np.linalg.solve` or
  `np.linalg.lstsq`, not by forming the inverse
- At the solution the residuals are uncorrelated with every feature and sum to zero
- Least squares is maximum likelihood under i.i.d. Gaussian noise; the fitted weights are unbiased,
  with covariance $\sigma^2(X^{\top}X)^{-1}$
- RMSE and $R^2$ measure a fit; measured on held-out data they estimate performance on new patients;
  residual plots check the noise model
### Wednesday, September 30 — Studio

You will fit the ten-feature model of section 6 by the normal equations, diagnose it through its
residuals and its weights, standardise the features, add a ridge penalty to the loss, and measure
what regularisation does when the cohort is small.

---
## Recommended resources

Everything here is optional and freely available. Each item covers material from this lecture in a
different style or voice; none of it is assigned.

### Start here

| Resource | Why this one | Time |
|---|---|---|
| **[The Main Ideas of Fitting a Line to Data](https://www.youtube.com/watch?v=PaFPbb66DxQ)** — StatQuest | Residuals, the sum of squares, and the bowl of section 3, with no matrices. | 9 min |
| **[Linear Regression, Clearly Explained](https://www.youtube.com/watch?v=nk2CQITm_eo)** — StatQuest | $R^2$, the residual sum of squares, and regression on several features, from a statistician's point of view. | 27 min |
| **[Regression Analysis](https://seeing-theory.brown.edu/regression-analysis/index.html)** — Seeing Theory, Brown University | Interactive: place the points and watch the least squares line and its residuals respond. | 15 min |
### The mathematics

| Resource | Why this one | Time |
|---|---|---|
| **[CS229 Lecture Notes](https://cs229.stanford.edu/main_notes.pdf)** — Ma and Ng, Stanford, Chapter 1 | The linear model, the normal equations, and least squares as maximum likelihood: the derivations of sections 4 and 5, written with $\theta$ in place of $w$. | 1 h |
| **[CS229 Lecture 2: Linear Regression and Gradient Descent](https://www.youtube.com/watch?v=4b4MUYve_U8)** — Stanford, Autumn 2018 | The lecture that accompanies the notes. Also introduces gradient descent, which is Meeting 13. | 78 min |
| **[Mathematics for Machine Learning](https://mml-book.github.io/)** — Deisenroth, Faisal and Ong, Chapter 9 | Linear regression as maximum likelihood, then with a prior on the weights, which is the ridge penalty of Wednesday's studio. Free PDF. | 2 h |
| **[Lecture 16: Projection Matrices and Least Squares](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/resources/lecture-16-projection-matrices-and-least-squares/)** — Gilbert Strang, MIT 18.06 | The geometric view of least squares, not covered in this lecture: the prediction $\hat{y}$ is the projection of $y$ onto the space spanned by the columns of $X$, and the normal equations say that the residual is perpendicular to that space. | 48 min |
### Textbooks, free PDFs

| Resource | Why this one | Time |
|---|---|---|
| **[An Introduction to Statistical Learning](https://www.statlearning.com/)** — James, Witten, Hastie and Tibshirani, Chapter 3 | The statistician's account: standard errors and confidence intervals for the weights, hypothesis tests, and the diagnostic plots of section 6. Editions with R and with Python. | 3 h |
| **[The Elements of Statistical Learning](https://hastie.su.domains/ElemStatLearn/)** — Hastie, Tibshirani and Friedman, Chapter 3 | The graduate-level version of the same chapter. Sections 3.1 and 3.2 correspond to this lecture; section 3.4 to ridge regression. | long |

### Source of the data

| Resource | Why this one |
|---|---|
| **[Least Angle Regression](https://doi.org/10.1214/009053604000000067)** — Efron, Hastie, Johnstone and Tibshirani, *Annals of Statistics*, 2004 | The paper that introduced the 442-patient diabetes dataset; section 1 describes the data. Open access. |
| **[The diabetes data](https://www4.stat.ncsu.edu/~boos/var.select/diabetes.html)** — North Carolina State University | The original data files and variable descriptions. |

### Looking ahead

| Resource | Why this one | Time |
|---|---|---|
| **[Regularization Part 1: Ridge (L2) Regression](https://www.youtube.com/watch?v=Q81RR3yKn30)** — StatQuest | Wednesday's studio, in twenty minutes. | 20 min |